# 02 - Limpieza y transformación

**Proyecto:** Analítica de ventas de videojuegos en Databricks  
**Autor:** Darren J. Blackman M.  
**Asignatura:** Análisis de Datos y Toma de Decisiones en Computación

## Objetivo
Aplicar limpieza, imputación, validación y transformación de variables. El resultado se guarda como **vgsales_limpio**.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

CATALOGO = spark.sql("SELECT current_catalog()").first()[0]
ESQUEMA = "default"

def tabla(nombre):
    return f"`{CATALOGO}`.`{ESQUEMA}`.`{nombre}`"

print(f"Catálogo activo: {CATALOGO}")
print(f"Esquema utilizado: {ESQUEMA}")

Catálogo activo: workspace
Esquema utilizado: default


In [0]:
TABLA_BRONZE = tabla("vgsales_bronze")
TABLA_LIMPIA = tabla("vgsales_limpio")
TABLA_CALIDAD = tabla("vgsales_calidad")

df = spark.table(TABLA_BRONZE)
filas_iniciales = df.count()
print(f"Registros antes de limpiar: {filas_iniciales:,}")

Registros antes de limpiar: 16,598


In [0]:
# Diagnóstico inicial
valores_faltantes = ["", "n/a", "na", "null", "none"]

nulos_antes = df.select([
    F.sum(
        F.when(
            F.col(c).isNull()
            | F.lower(F.trim(F.col(c).cast("string"))).isin(valores_faltantes),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df.columns
])

display(nulos_antes)

filas_duplicadas = df.count() - df.dropDuplicates().count()
print(f"Duplicados exactos: {filas_duplicadas:,}")

Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,0,0,271,0,58,0,0,0,0,0


Duplicados exactos: 0


In [0]:
# Normalización de textos
columnas_texto = ["Name", "Platform", "Genre", "Publisher"]

for c in columnas_texto:
    df = df.withColumn(c, F.trim(F.col(c).cast("string")))

columnas_ventas = [
    "NA_Sales",
    "EU_Sales",
    "JP_Sales",
    "Other_Sales",
    "Global_Sales"
]

# try_cast convierte N/A en NULL sin detener la ejecución
df_limpio = (
    df
    .withColumn("Year", F.expr("try_cast(`Year` AS INT)"))
    .withColumn("Rank", F.expr("try_cast(`Rank` AS INT)"))
    .withColumn(
        "Publisher",
        F.when(
            F.col("Publisher").isNull()
            | F.lower(F.trim(F.col("Publisher"))).isin(
                "", "n/a", "na", "null", "none"
            ),
            F.lit("Desconocido")
        ).otherwise(F.col("Publisher"))
    )
    .filter(F.col("Year").isNotNull())
)

for c in columnas_ventas:
    df_limpio = df_limpio.withColumn(
        c,
        F.expr(f"try_cast(`{c}` AS DOUBLE)")
    )

df_limpio = df_limpio.dropDuplicates()

In [0]:
# Validaciones de calidad
condicion_ventas_validas = F.lit(True)
for c in columnas_ventas:
    condicion_ventas_validas = condicion_ventas_validas & (F.col(c) >= 0)

df_limpio = df_limpio.filter(condicion_ventas_validas)

# Variables derivadas
df_limpio = (
    df_limpio
    .withColumn("Decada", (F.floor(F.col("Year") / 10) * 10).cast("int"))
    .withColumn(
        "Region_Dominante",
        F.when(F.col("NA_Sales") == F.greatest("NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"), "Norteamérica")
         .when(F.col("EU_Sales") == F.greatest("NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"), "Europa")
         .when(F.col("JP_Sales") == F.greatest("NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales"), "Japón")
         .otherwise("Otras regiones")
    )
)

In [0]:
filas_finales = df_limpio.count()
registros_eliminados = filas_iniciales - filas_finales

nulos_despues = df_limpio.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_limpio.columns
])

resumen_calidad = spark.createDataFrame([
    ("Registros iniciales", filas_iniciales),
    ("Registros finales", filas_finales),
    ("Registros eliminados", registros_eliminados),
    ("Duplicados detectados", filas_duplicadas),
], ["Metrica", "Valor"])

display(resumen_calidad)
display(nulos_despues)

Metrica,Valor
Registros iniciales,16598
Registros finales,16327
Registros eliminados,271
Duplicados detectados,0


Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Decada,Region_Dominante
0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
# Persistencia de la capa limpia y del resumen de calidad
(
    df_limpio.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_LIMPIA)
)

(
    resumen_calidad.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABLA_CALIDAD)
)

print(f"Tabla limpia creada: {TABLA_LIMPIA}")
print(f"Registros finales: {filas_finales:,}")

Tabla limpia creada: `workspace`.`default`.`vgsales_limpio`
Registros finales: 16,327


In [0]:
from pyspark.sql import functions as F

CATALOGO = spark.sql("SELECT current_catalog()").first()[0]

df_antes = spark.table(f"`{CATALOGO}`.`default`.`vgsales_bronze`")
df_despues = spark.table(f"`{CATALOGO}`.`default`.`vgsales_limpio`")

valores_faltantes = ["", "n/a", "na", "null", "none"]

def contar_faltantes(dataframe, columnas):
    return dataframe.select([
        F.sum(
            F.when(
                F.col(c).isNull()
                | F.lower(F.trim(F.col(c).cast("string"))).isin(valores_faltantes),
                1
            ).otherwise(0)
        ).alias(c)
        for c in columnas
    ])

columnas = df_antes.columns

conteo_antes = contar_faltantes(df_antes, columnas).first().asDict()
conteo_despues = contar_faltantes(
    df_despues,
    columnas
).first().asDict()

comparacion = spark.createDataFrame(
    [
        (
            columna,
            int(conteo_antes.get(columna, 0) or 0),
            int(conteo_despues.get(columna, 0) or 0)
        )
        for columna in columnas
    ],
    ["Columna", "Nulos_antes", "Nulos_despues"]
)

display(comparacion)

Columna,Nulos_antes,Nulos_despues
Rank,0,0
Name,0,0
Platform,0,0
Year,271,0
Genre,0,0
Publisher,58,0
NA_Sales,0,0
EU_Sales,0,0
JP_Sales,0,0
Other_Sales,0,0


In [0]:
from pyspark.sql import functions as F

CATALOGO = spark.sql("SELECT current_catalog()").first()[0]

df_limpio = spark.table(
    f"`{CATALOGO}`.`default`.`vgsales_limpio`"
)

display(
    df_limpio.select(
        "Rank",
        "Name",
        "Platform",
        "Year",
        "Publisher",
        "Decada",
        "Region_Dominante",
        "Global_Sales"
    ).limit(10)
)

Rank,Name,Platform,Year,Publisher,Decada,Region_Dominante,Global_Sales
7,New Super Mario Bros.,DS,2006,Nintendo,2000,Norteamérica,30.01
13,Pokemon Gold/Pokemon Silver,GB,1999,Nintendo,1990,Norteamérica,23.1
18,Grand Theft Auto: San Andreas,PS2,2004,Take-Two Interactive,2000,Otras regiones,20.81
20,Brain Age: Train Your Brain in Minutes a Day,DS,2005,Nintendo,2000,Europa,20.22
46,Pokemon HeartGold/Pokemon SoulSilver,DS,2009,Nintendo,2000,Norteamérica,11.9
80,Halo 2,XB,2004,Microsoft Game Studios,2000,Norteamérica,8.49
91,Grand Theft Auto: Liberty City Stories,PSP,2005,Take-Two Interactive,2000,Norteamérica,7.72
101,The Legend of Zelda: Twilight Princess,Wii,2006,Nintendo,2000,Norteamérica,7.31
105,Need for Speed Underground,PS2,2003,Electronic Arts,2000,Norteamérica,7.2
113,FIFA 14,PS3,2013,Electronic Arts,2010,Europa,6.9
